# Pipeline Crawling Data Piksel Sentinel-2A via openEO CDSE (Jawa Timur)
## Ekstraksi Spektral Multiband (B02, B03, B04, B08, B11, NDVI, NDWI, MNDWI, NDBI) per Piksel Poligon

---

### Deskripsi Alur Kerja
Notebook ini mengeksekusi pipeline pengunduhan dan pemrosesan piksel satelit multispektral skala besar dari **Copernicus Data Space Ecosystem (CDSE) openEO**:
1. **Membaca Sampel Poligon:** Memuat 250 poligon sampel hasil digitasi (5 kelas @ 50 poligon).
2. **Tiling Grid Spasial:** Membagi poligon sampel ke dalam grid *tile* kecil (~22 km x 22 km) agar lolos batas ukuran pemrosesan openEO dan meminimalisir kuota.
3. **Pengiriman Batch Job:** Mengirim job pemrosesan ke server openEO CDSE dengan masking awan berbasis band SCL (*Scene Classification Layer*).
4. **Ekstraksi Piksel & Indeks:** Mengunduh GeoTIFF hasil job, mengekstraksi piksel di dalam poligon sampel, menghitung indeks spektral (`NDVI`, `NDWI`, `MNDWI`, `NDBI`), dan mengekspor hasilnya ke file CSV `piksel_s2.csv`.

In [ ]:
# 1. Import Dependensi Utama
import os
import time
import threading
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask as rio_mask
from rasterio.features import rasterize
from rasterio.warp import transform as warp_transform
from shapely.geometry import box
from shapely.validation import make_valid

import openeo
from openeo.extra.job_management import CsvJobDatabase, MultiBackendJobManager

print("[OK] Semua library dependensi berhasil di-import!")

In [ ]:
# 2. Konfigurasi Parameter & Deteksi Path Otomatis
# (Mendeteksi direktori dengan aman baik dijalankan dari folder 'uts/code' maupun root workspace)
cwd = Path.cwd()
if (cwd / "uts" / "data").exists():
    ROOT_UTS = cwd / "uts"
elif (cwd.parent / "data").exists() and cwd.name == "code":
    ROOT_UTS = cwd.parent
elif (cwd / "data").exists():
    ROOT_UTS = cwd
else:
    ROOT_UTS = Path("../").resolve()

GEOJSON_DIR = ROOT_UTS / "data" / "geojson"
OUT_DIR = ROOT_UTS / "data" / "csv" / "hasil_s2_piksel"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Definisi Sumber File Sampel Poligon (Mendukung .zip maupun .geojson)
SUMBER = {}
kelas_config = [
    ("Sawah", 1, "sawah.zip", "jawatimur_Sawah.geojson"),
    ("Pemukiman", 2, "pemukiman.zip", "jawatimur_Pemukiman.geojson"),
    ("Hutan Non-Mangrove", 3, "hutan_non_mangrove.zip", "jawatimur_Hutan_non_mangrove.geojson"),
    ("Air", 4, "air.zip", "jawatimur_Air.geojson"),
    ("Hutan Mangrove", 5, "hutan_mangrove.zip", "jawatimur_Hutan_mangrove.geojson"),
]

for nama_kelas, kode_label, zip_name, geojson_name in kelas_config:
    zip_path = GEOJSON_DIR / zip_name
    geojson_path = GEOJSON_DIR / geojson_name
    if zip_path.exists():
        SUMBER[nama_kelas] = (str(zip_path), kode_label)
    elif geojson_path.exists():
        SUMBER[nama_kelas] = (str(geojson_path), kode_label)
    else:
        SUMBER[nama_kelas] = (str(zip_path), kode_label)

# Parameter Grid Tiling & Satelit
TILE_DEG = 0.20          # Ukuran grid tile ~22 km x 22 km (~2200x2200 piksel @10m)
BUFFER_DEG = 0.005       # Buffer margin ~500 meter di sekeliling batas poligon
TANGGAL = ["2024-08-01", "2024-08-31"]   # Rentang waktu observasi musim kemarau (minim awan)
BANDS = ["B02", "B03", "B04", "B08", "B11"]  # Band spektral yang diunduh
PAKAI_MASK_SCL = True    # Masking awan/bayangan per piksel via band SCL
MAX_CLOUD = 30           # Batas toleransi tutupan awan scene (%)
PARALEL = 2              # Jumlah job konkuren (akun CDSE gratis max 2)

# Parameter Ekstraksi Piksel
MAKS_PIKSEL_PER_POLIGON = 2000  # Kuota maksimal piksel per poligon sampel (mencegah file CSV membengkak)
SEED = 42
BUAT_RASTER_LABEL = True        # Buat label_<tile>.tif di folder masing-masing job

JOB_DB_PATH = OUT_DIR / "daftar_job.csv"
PETA_CSV = OUT_DIR / "peta_sampel_tile.csv"
PIKSEL_OUT = OUT_DIR / "piksel_s2.csv"

print(f"Direktori GeoJSON : {GEOJSON_DIR}")
print(f"Direktori Output  : {OUT_DIR}")
print(f"Target Output CSV : {PIKSEL_OUT}")

---
## Step 1: Memuat & Menggabungkan Poligon Sampel 5 Kelas Tutupan Lahan

In [ ]:
def baca_semua() -> gpd.GeoDataFrame:
    """Membaca semua file poligon sampel dan menggabungkannya ke dalam GeoDataFrame WGS84."""
    bagian = []
    for nama, (file_path, label) in SUMBER.items():
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File sampel untuk '{nama}' tidak ditemukan di: {file_path}")
            
        g = gpd.read_file(file_path).to_crs("EPSG:4326")
        g["poligon_no"] = np.arange(len(g))
        g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()
        g["geometry"] = g.geometry.apply(lambda x: x if x.is_valid else make_valid(x))
        g["label_teks"] = nama
        g["label"] = label
        g["poligon_id"] = [f"{nama}_{n:04d}" for n in g["poligon_no"]]
        g = g[["poligon_id", "poligon_no", "label", "label_teks", "geometry"]]
        print(f"Kelas: {nama:<18} | Jumlah: {len(g):>3} poligon | Bounds: {[round(v, 2) for v in g.total_bounds]}")
        bagian.append(g)

    gdf = gpd.GeoDataFrame(pd.concat(bagian, ignore_index=True), crs="EPSG:4326")
    print("-" * 60)
    print(f"Total Keseluruhan Sampel: {len(gdf)} Poligon")
    return gdf

gdf_sampel = baca_semua()
gdf_sampel.head(3)

---
## Step 2: Pembagian Grid Tile Spasial (*Tiling System*)

Poligon sampel dipecah dan dipetakan ke dalam grid tile `x{gx}_y{gy}` berukuran 0.20° (~22 km) agar openEO memproses tiap area secara modular tanpa memicu *memory limit*.

In [ ]:
def buat_tile(gdf: gpd.GeoDataFrame):
    T, B = TILE_DEG, BUFFER_DEG
    peta, batas, sel_xy = [], {}, {}

    for idx, geom in gdf.geometry.items():
        minx, miny, maxx, maxy = geom.bounds
        for gx in range(int(np.floor(minx / T)), int(np.floor(maxx / T)) + 1):
            for gy in range(int(np.floor(miny / T)), int(np.floor(maxy / T)) + 1):
                sel = box(gx * T, gy * T, (gx + 1) * T, (gy + 1) * T)
                if not geom.intersects(sel):
                    continue
                bagian = geom.intersection(sel)
                if bagian.is_empty:
                    continue
                b = bagian.bounds
                tid = f"x{gx}_y{gy}"
                peta.append({"tile_id": tid, "sampel_id": idx, "poligon_id": gdf.at[idx, "poligon_id"]})
                sel_xy[tid] = (gx, gy)
                if tid in batas:
                    o = batas[tid]
                    batas[tid] = (min(o[0], b[0]), min(o[1], b[1]), max(o[2], b[2]), max(o[3], b[3]))
                else:
                    batas[tid] = b

    tiles = pd.DataFrame([
        {
            "tile_id": tid,
            "gx": sel_xy[tid][0],
            "gy": sel_xy[tid][1],
            "west": b[0] - B,
            "south": b[1] - B,
            "east": b[2] + B,
            "north": b[3] + B,
        }
        for tid, b in batas.items()
    ])
    peta = pd.DataFrame(peta)
    tiles["n_sampel"] = tiles["tile_id"].map(peta.groupby("tile_id")["sampel_id"].nunique())

    lebar_km = (tiles.east - tiles.west) * 111.32 * np.cos(np.radians((tiles.north + tiles.south) / 2))
    tinggi_km = (tiles.north - tiles.south) * 110.57
    print(f"Total Grid Tile (Job)   : {len(tiles)} Tile")
    print(f"Ukuran Tile Terbesar    : {lebar_km.max():.1f} km x {tinggi_km.max():.1f} km")
    print(f"Total Estimasi Area     : {(lebar_km * tinggi_km).sum():,.0f} km²")
    return tiles, peta

tiles_df, peta_df = buat_tile(gdf_sampel)
tiles_df.head(3)

---
## Step 3: Pengiriman & Pemantauan Batch Job openEO CDSE

Koneksi otomatis ke `https://openeo.dataspace.copernicus.eu` menggunakan protokol OIDC. Status progres tiap job disimpan ke `daftar_job.csv` sehingga jika terjadi interupsi, job yang sudah selesai tidak akan diulang kembali.

In [ ]:
def _jam() -> str:
    return datetime.now().strftime("%H:%M:%S")

def start_job(row, connection, **kwargs):
    print(f"[{_jam()}] >> Mengirim job tile {row['tile_id']} ({row.get('n_sampel', '?')} poligon)...", flush=True)
    bbox = {
        "west": float(row["west"]),
        "south": float(row["south"]),
        "east": float(row["east"]),
        "north": float(row["north"]),
    }
    bands = BANDS + ["SCL"] if PAKAI_MASK_SCL else BANDS
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=TANGGAL,
        bands=bands,
        max_cloud_cover=MAX_CLOUD,
    )

    if PAKAI_MASK_SCL:
        scl = cube.band("SCL")
        # Masking piksel jenuh, bayangan awan, awan sedang/tinggi, dan cirrus
        mask_awan = (scl == 1) | (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10)
        cube = cube.filter_bands(BANDS).mask(mask_awan)

    komposit = cube.median_time()
    return komposit.create_job(title=f"s2_{row['tile_id']}", out_format="GTiff")

class ManagerBerlapor(MultiBackendJobManager):
    def on_job_done(self, job, row):
        print(f"[{_jam()}] [OK] Tile {row['tile_id']} selesai (job {job.job_id}), mengunduh hasil...", flush=True)
        super().on_job_done(job, row)
        print(f"[{_jam()}] [OK] Tile {row['tile_id']} berhasil terunduh.", flush=True)

    def on_job_error(self, job, row):
        print(f"[{_jam()}] [ERROR] Tile {row['tile_id']} gagal (job {job.job_id}).", flush=True)
        super().on_job_error(job, row)

def _lapor_berkala(job_db, berhenti: threading.Event, interval: int = 30):
    mulai = time.time()
    while not berhenti.wait(interval):
        try:
            df = job_db.read()
        except Exception:
            continue
        n = len(df)
        hitung = df["status"].value_counts().to_dict()
        selesai = hitung.get("finished", 0)
        gagal = sum(hitung.get(s, 0) for s in ("error", "start_failed", "skipped"))
        jalan = hitung.get("running", 0) + hitung.get("queued", 0) + hitung.get("created", 0)
        belum = hitung.get("not_started", 0)
        persen = 100 * selesai / n if n else 0
        menit = (time.time() - mulai) / 60
        print(f"[{_jam()}] Progres: {selesai}/{n} ({persen:.0f}%) | Berjalan/Antre: {jalan} | Belum: {belum} | Gagal: {gagal} | {menit:.1f} mnt", flush=True)

def jalankan_job(tiles: pd.DataFrame, ulangi_error: bool = True):
    conn = openeo.connect("https://openeo.dataspace.copernicus.eu")
    conn.authenticate_oidc()

    manager = ManagerBerlapor(root_dir=OUT_DIR, poll_sleep=30)
    manager.add_backend("cdse", connection=conn, parallel_jobs=PARALEL)

    job_db = CsvJobDatabase(JOB_DB_PATH)
    if not job_db.exists():
        job_db.initialize_from_df(tiles)
    elif ulangi_error:
        df = job_db.read()
        gagal = df["status"].isin(["error", "start_failed", "skipped"])
        if gagal.any():
            print(f"Mengulang {int(gagal.sum())} job yang sebelumnya gagal.")
            df.loc[gagal, "status"] = "not_started"
            job_db.persist(df)

    berhenti = threading.Event()
    pemantau = threading.Thread(target=_lapor_berkala, args=(job_db, berhenti), daemon=True)
    pemantau.start()
    try:
        manager.run_jobs(start_job=start_job, job_db=job_db)
    finally:
        berhenti.set()
        pemantau.join(timeout=5)

    df = job_db.read()
    print("\n=== RINGKASAN STATUS JOB OPENEO ===")
    print(df["status"].value_counts().to_string())
    return df

---
## Step 4: Ekstraksi Piksel Berlabel & Kalkulasi Indeks Spektral

Membaca GeoTIFF tiap tile hasil unduhan, mengekstraksi piksel yang beririsan dengan poligon sampel, menghitung nilai `NDVI`, `NDWI`, `MNDWI`, `NDBI`, dan menyimpan dataset final ke `piksel_s2.csv`.

In [ ]:
def _nama_band(src) -> list:
    desc = list(src.descriptions or [])
    if len(desc) == src.count and all(desc) and set(BANDS) <= set(desc):
        return desc
    return BANDS

def _valid_mask(data, nodata, masked=None):
    ok = np.isfinite(data).all(axis=0) & (data > 0).all(axis=0)
    if nodata is not None and not np.isnan(nodata):
        ok &= ~(data == nodata).any(axis=0)
    if masked is not None:
        ok &= ~masked
    return ok

def _piksel_geom(src, geom):
    nodata = src.nodata
    if geom.geom_type == "Point":
        rr, cc = src.index(geom.x, geom.y)
        if not (0 <= rr < src.height and 0 <= cc < src.width):
            return None
        data = src.read(window=Window(cc, rr, 1, 1)).astype("float64")
        ok = _valid_mask(data, nodata)
        row0, col0 = rr, cc
    else:
        ok = None
        for all_touched in (False, True):
            try:
                arr, tr = rio_mask(src, [geom], crop=True, all_touched=all_touched, filled=False)
            except ValueError:
                return None
            data = np.ma.getdata(arr).astype("float64")
            ok = _valid_mask(data, nodata, np.ma.getmaskarray(arr).any(axis=0))
            row0 = int(round((tr.f - src.transform.f) / src.transform.e))
            col0 = int(round((tr.c - src.transform.c) / src.transform.a))
            if ok.any():
                break
        if ok is None or not ok.any():
            return None

    r_idx, c_idx = np.nonzero(ok)
    if len(r_idx) == 0:
        return None
    nilai = data[:, r_idx, c_idx]
    px_row, px_col = row0 + r_idx, col0 + c_idx
    xs, ys = rasterio.transform.xy(src.transform, px_row, px_col)
    return nilai, px_row, px_col, np.asarray(xs), np.asarray(ys)

def tambah_indeks(df: pd.DataFrame) -> pd.DataFrame:
    e = 1e-9
    df["NDVI"] = (df["B08"] - df["B04"]) / (df["B08"] + df["B04"] + e)
    df["NDWI"] = (df["B03"] - df["B08"]) / (df["B03"] + df["B08"] + e)
    df["MNDWI"] = (df["B03"] - df["B11"]) / (df["B03"] + df["B11"] + e)
    df["NDBI"] = (df["B11"] - df["B08"]) / (df["B11"] + df["B08"] + e)
    return df

def _tif_tile(r):
    tifs = sorted(p for p in OUT_DIR.glob(f"job_{r['id']}/*.tif") if not p.name.startswith("label_"))
    return tifs[0] if tifs else None

def ekstrak_piksel(gdf, db, peta) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    T = TILE_DEG
    potongan = []
    selesai = db[db["status"] == "finished"]
    print(f"\nMengekstrak piksel dari {len(selesai)} tile yang telah selesai...")

    total_tile = len(selesai)
    total_piksel = 0
    t0 = time.time()

    for no_tile, (_, r) in enumerate(selesai.iterrows(), start=1):
        tif = _tif_tile(r)
        if tif is None:
            continue

        cx0, cy0 = int(r["gx"]) * T, int(r["gy"]) * T
        ids = peta.loc[peta["tile_id"] == r["tile_id"], "sampel_id"].unique()
        n_poli = len(ids)
        piksel_tile = 0

        with rasterio.open(tif) as src:
            nama_band = _nama_band(src)
            sub_r = gdf.loc[ids].to_crs(src.crs)

            for no_poli, (idx, row) in enumerate(sub_r.iterrows(), start=1):
                hasil = _piksel_geom(src, row.geometry)
                if hasil is None:
                    continue
                nilai, px_row, px_col, xs, ys = hasil

                lon, lat = warp_transform(src.crs, "EPSG:4326", xs.tolist(), ys.tolist())
                lon, lat = np.asarray(lon), np.asarray(lat)

                pilih = np.flatnonzero((lon >= cx0) & (lon < cx0 + T) & (lat >= cy0) & (lat < cy0 + T))
                if len(pilih) == 0:
                    continue
                if MAKS_PIKSEL_PER_POLIGON and len(pilih) > MAKS_PIKSEL_PER_POLIGON:
                    pilih = rng.choice(pilih, MAKS_PIKSEL_PER_POLIGON, replace=False)

                d = {
                    "poligon_id": row["poligon_id"],
                    "poligon_no": row["poligon_no"],
                    "sampel_id": idx,
                    "tile_id": r["tile_id"],
                    "label": row["label"],
                    "label_teks": row["label_teks"],
                    "px_row": px_row[pilih],
                    "px_col": px_col[pilih],
                    "lon": lon[pilih],
                    "lat": lat[pilih],
                }
                for i, nb in enumerate(nama_band):
                    d[nb] = nilai[i, pilih]
                potongan.append(pd.DataFrame(d))
                piksel_tile += len(pilih)
                total_piksel += len(pilih)

        print(f"[{_jam()}] Tile {no_tile}/{total_tile} ({r['tile_id']}): {piksel_tile:,} piksel diekstrak", flush=True)

    if not potongan:
        print("Tidak ada piksel yang berhasil diekstrak.")
        return pd.DataFrame()

    df = pd.concat(potongan, ignore_index=True)
    kunci = ["tile_id", "px_row", "px_col"]
    df = df.drop_duplicates(kunci).reset_index(drop=True)
    df = tambah_indeks(df)

    df.to_csv(PIKSEL_OUT, index=False)
    print(f"\n[SUKSES] Dataset piksel tersimpan ke: {PIKSEL_OUT}")
    print(f"Total Piksel Berlabel: {len(df):,}")
    print(df.groupby("label_teks").size().to_string())
    return df

---
## Step 5: Eksekusi Pipeline Crawling & Ekstraksi Piksel

Jalankan fungsi utama berikut:
- `jalankan=True`: Kirim dan pantau batch job ke server CDSE openEO.
- `ekstrak=True`: Ekstrak piksel dari raster tile yang telah selesai.
- Jika job sudah pernah diunduh sebelumnya, set `jalankan=False` untuk langsung mengekstrak CSV dari data tile yang ada.

In [ ]:
def main(jalankan: bool = True, ekstrak: bool = True, ulangi_error: bool = True):
    gdf = baca_semua()
    tiles, peta = buat_tile(gdf)

    # Simpan file rencana tile dan peta sampel
    tiles.to_csv(OUT_DIR / "rencana_tile.csv", index=False)
    peta.to_csv(PETA_CSV, index=False)
    gdf.drop(columns="geometry").assign(luas_derajat2=gdf.geometry.area).to_csv(OUT_DIR / "daftar_poligon.csv", index_label="sampel_id")

    if jalankan:
        db = jalankan_job(tiles, ulangi_error=ulangi_error)
    else:
        db = CsvJobDatabase(JOB_DB_PATH).read()

    hasil = None
    if ekstrak:
        hasil = ekstrak_piksel(gdf, db, peta)
    return hasil if hasil is not None else db

# EKSEKUSI PIPELINE (Ubah jalankan=True jika ingin memulai request openEO baru)
# df_piksel = main(jalankan=True, ekstrak=True)

---
## Step 6: Pratinjau & Validasi Data Piksel yang Berhasil Disimpan

In [ ]:
if PIKSEL_OUT.exists():
    df_hasil = pd.read_csv(PIKSEL_OUT)
    print(f"File: {PIKSEL_OUT}")
    print(f"Dimensi Data: {df_hasil.shape[0]:,} baris x {df_hasil.shape[1]} kolom")
    print("\nDistribusi Piksel per Kelas Tutupan Lahan:")
    print(df_hasil['label_teks'].value_counts())
    display(df_hasil.head(5))
else:
    print(f"File {PIKSEL_OUT} belum terbentuk. Jalankan main() untuk mengekstrak data!")